In [24]:
import copy
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from functions import functions

In [ ]:
print(f"Screening Candidate Levels on: {device}\n")
train_base, test_base = get_mnist_base()
test_loader = DataLoader(test_base, batch_size=512, shuffle=False)

# Center-point baseline held constant during screening
center_config = {
    "num_images": 10000,
    "noise_rate": 0.0,
    "imbalance_ratio": 1.0,
    "lr": 1e-3,
    "batch_size": 128,
    "dropout_rate": 0.2,
    "hidden_layers": 2,
    "nodes_per_layer": 128,
}

# Candidate grid: test 4-5 levels per factor to observe response curves
candidate_grid = {
    "num_images": [1000, 2500, 5000, 15000, 30000],
    "noise_rate": [0.0, 0.15, 0.30, 0.45, 0.60],
    "imbalance_ratio": [1.0, 5.0, 10.0, 25.0, 50.0],
    "lr": [1e-4, 5e-4, 1e-3, 5e-3, 2e-2],
    "batch_size": [16, 64, 128, 512, 1024],
    "dropout_rate": [0.0, 0.2, 0.4, 0.6, 0.75],
    "hidden_layers": [1, 2, 3, 5, 7],
    "nodes_per_layer": [16, 32, 64, 256, 512],
}

recommended_design = {}

for param, candidates in candidate_grid.items():
    print(f"--- Screening Factor: {param} ---")
    results = []
    for val in candidates:
        cfg = copy.deepcopy(center_config)
        cfg[param] = val
        metrics = evaluate_config_detailed(train_base, test_loader, cfg, epochs=10)
        results.append((val, metrics))
        min_cls_str = (
            f" | Downsampled: {metrics['minority_classes']}"
            if metrics["minority_classes"]
            else ""
        )
        print(
            f"  {param} = {str(val):<8} -> "
            f"Acc: {metrics['balanced_acc']:6.2f}% | "
            f"MinRec: {metrics['minority_recall']:6.2f}% | "
            f"MacroF1: {metrics['macro_f1']:6.2f}% | "
            f"Loss: {metrics['test_loss']:.4f}{min_cls_str}"
        )

    low_pair, high_pair = recommend_binary_levels(results, primary_metric="macro_f1")
    low_val, low_m = low_pair
    high_val, high_m = high_pair

    recommended_design[param] = {
        "-1": low_val,
        "+1": high_val,
        "acc_spread": high_m["balanced_acc"] - low_m["balanced_acc"],
        "minrec_spread": high_m["minority_recall"] - low_m["minority_recall"],
        "f1_spread": high_m["macro_f1"] - low_m["macro_f1"],
        "loss_spread": high_m["test_loss"] - low_m["test_loss"],
    }
    print(
        f"  >> Recommended Pair: {low_val} vs. {high_val} | "
        f"Spreads -> Acc: {recommended_design[param]['acc_spread']:+.2f}%, "
        f"MinRec: {recommended_design[param]['minrec_spread']:+.2f}%, "
        f"MacroF1: {recommended_design[param]['f1_spread']:+.2f}%, "
        f"Loss: {recommended_design[param]['loss_spread']:+.4f}\n"
    )

# Summary Table for your 2^(k-p) Fractional Factorial Design
print("=" * 108)
print("SUGGESTED 2-LEVEL FRACTIONAL FACTORIAL LEVELS (-1 vs +1)")
print("=" * 108)
print(
    f"{'Factor':<18} | {'Level -1':<10} | {'Level +1':<10} | "
    f"{'Acc Spread':<12} | {'MinRec Spread':<14} | {'MacroF1 Spread':<15} | {'Loss Spread':<12}"
)
print("-" * 108)
for param, info in recommended_design.items():
    print(
        f"{param:<18} | {str(info['-1']):<10} | {str(info['+1']):<10} | "
        f"{info['acc_spread']:>+10.2f}% | {info['minrec_spread']:>+12.2f}% | "
        f"{info['f1_spread']:>+13.2f}% | {info['loss_spread']:>+11.4f}"
    )

Screening Candidate Levels on: cuda

--- Screening Factor: num_images ---


/tmp/ipykernel_1348/565992359.py:14: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  targets = np.array(base_dataset.targets)


  num_images = 1000     -> Acc:  88.28% | MinRec:  88.28% | MacroF1:  88.32% | Loss: 0.3692
  num_images = 2500     -> Acc:  91.83% | MinRec:  91.83% | MacroF1:  91.88% | Loss: 0.2620
  num_images = 5000     -> Acc:  93.85% | MinRec:  93.85% | MacroF1:  93.84% | Loss: 0.1988
  num_images = 15000    -> Acc:  96.35% | MinRec:  96.35% | MacroF1:  96.35% | Loss: 0.1178
  num_images = 30000    -> Acc:  96.90% | MinRec:  96.90% | MacroF1:  96.92% | Loss: 0.1105
  >> Recommended Pair: 1000 vs. 30000 | Spreads -> Acc: +8.62%, MinRec: +8.62%, MacroF1: +8.59%, Loss: -0.2586

--- Screening Factor: noise_rate ---
  noise_rate = 0.0      -> Acc:  95.52% | MinRec:  95.52% | MacroF1:  95.50% | Loss: 0.1462
  noise_rate = 0.15     -> Acc:  94.67% | MinRec:  94.67% | MacroF1:  94.67% | Loss: 0.2806
  noise_rate = 0.3      -> Acc:  92.91% | MinRec:  92.91% | MacroF1:  92.89% | Loss: 0.6032
  noise_rate = 0.45     -> Acc:  91.32% | MinRec:  91.32% | MacroF1:  91.32% | Loss: 0.8513
  noise_rate = 0.6     